# XGBoost

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd

import xgboost as xgb

In [2]:
SEED = 10

In [3]:
np.random.seed(SEED)
_ = torch.manual_seed(SEED)

## Dataset

In [4]:
from dataset_PropertyPrice import DatasetHousePrice

ds_name = 'house_price_traj'
ds_df_dir = '/home/yc366/repos/survsurf_benchmark/dataset_split'
project = 'SurvSurfBenchmark_HousePrice'
SAVE_PATH = f'./xgboost_models/{project}/xgboost_mono_{SEED}'
g_resol = 0.01
t_resol = 1
split = 'train'
ds = DatasetHousePrice(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    split=split, 
    mode='first_cross_obs_only', 
    separate_g_from_feats=False
)
df_train = ds._get_df_Xy_trans_obs()

split = 'train'
ds = DatasetHousePrice(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    split=split, 
    mode='true_probs_grid_naless', 
    separate_g_from_feats=False
)
df_train_pred = ds._get_df_Xy_true_prob(dropna=True)

split = 'val'
ds = DatasetHousePrice(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    split=split, 
    mode='true_probs_grid_naless', 
    separate_g_from_feats=False
)
df_val = ds._get_df_Xy_true_prob(dropna=True)

In [5]:
df_train.head()

,traj_id,event_observed,duration,g_max_by_time,feat__is_new_build,feat__long,feat__lat,feat__frac_properties_n_beds__1,feat__frac_properties_n_beds__2,feat__frac_properties_n_beds__3,...,feat__deprived_4_dim,feat__soc_grade_AB,feat__soc_grade_C1,feat__soc_grade_C2,feat__soc_grade_DE,feat__prop_type_Detached,feat__prop_type_SemiD,feat__prop_type_Terraced,weight,is_t_trans
0,E06000001DetachedExisting,1,1.0,0.000580,0.0,0.021889,2.09093,-0.553782,-0.876423,-0.709005,...,0.088157,-1.329559,-1.741656,0.500371,1.905991,1.0,0.0,0.0,1,1
1,E06000001DetachedExisting,1,3.0,0.008855,0.0,0.021889,2.09093,-0.553782,-0.876423,-0.709005,...,0.088157,-1.329559,-1.741656,0.500371,1.905991,1.0,0.0,0.0,1,1
2,E06000001DetachedExisting,1,4.0,0.035069,0.0,0.021889,2.09093,-0.553782,-0.876423,-0.709005,...,0.088157,-1.329559,-1.741656,0.500371,1.905991,1.0,0.0,0.0,1,1
3,E06000001DetachedExisting,1,5.0,0.063205,0.0,0.021889,2.09093,-0.553782,-0.876423,-0.709005,...,0.088157,-1.329559,-1.741656,0.500371,1.905991,1.0,0.0,0.0,1,1
4,E06000001DetachedExisting,1,6.0,0.206230,0.0,0.021889,2.09093,-0.553782,-0.876423,-0.709005,...,0.088157,-1.329559,-1.741656,0.500371,1.905991,1.0,0.0,0.0,1,1


## Feature transforms

In [6]:
x_train = df_train.loc[:,df_train.columns.str.startswith('feat')|df_train.columns.str.startswith('g_max_by_time')].astype('float32')
x_train['g_max_by_time'] = x_train['g_max_by_time']/ds.g_max
mono_constraints = {i:0 if i != 'g_max_by_time' else -1 for i in x_train.columns}
x_train = x_train


x_train_pred = df_train_pred.loc[:,df_train_pred.columns.str.startswith('feat')|df_train_pred.columns.str.startswith('g_max_by_time')].astype('float32')
x_train_pred['g_max_by_time'] = x_train_pred['g_max_by_time']/ds.g_max
x_train_pred = x_train_pred


x_val = df_val.loc[:,df_val.columns.str.startswith('feat')|df_val.columns.str.startswith('g_max_by_time')].astype('float32')
x_val['g_max_by_time'] = x_val['g_max_by_time']/ds.g_max
x_val = x_val

assert all(df_train.loc[:,df_train.columns.str.startswith('feat')|df_train.columns.str.startswith('g_max_by_time')].columns == (
    df_val.loc[:,df_val.columns.str.startswith('feat')|df_val.columns.str.startswith('g_max_by_time')].columns
))



In [7]:
import xgboost as xgb
y_train = df_train['duration']*np.where(df_train['event_observed']==1, 1, -1)


params = {
    'objective': 'survival:cox',
    'monotone_constraints': mono_constraints,
    'random_state':10
}
dtrain = xgb.DMatrix(x_train, label=y_train)
xgboost_model = xgb.train(params, dtrain)
from lifelines import CoxPHFitter

train_results = pd.DataFrame({
    'xgb_risk': np.log(xgboost_model.predict(dtrain)),
    'time': df_train['duration'],
    'event': df_train['event_observed']
})

# Fit a simple Cox model on the XGBoost scores to get the baseline hazard
cph = CoxPHFitter()
cph.fit(train_results, duration_col='time', event_col='event')


<lifelines.CoxPHFitter: fitted with 10717 total observations, 1897 right-censored observations>

## Prediction

In [8]:
df_val['event_observed'].value_counts()

event_observed
0.0    7163
1.0    6118
Name: count, dtype: int64

In [9]:
df_train_pred['event_observed'].value_counts()

event_observed
0.0    68348
1.0    58418
Name: count, dtype: int64

In [10]:
dval = xgb.DMatrix(x_val)
# dtrain = xgb.DMatrix(x_train, label=y_train)
from lifelines import CoxPHFitter

val_results = pd.DataFrame({
    'xgb_risk': np.log(xgboost_model.predict(dval)),
})
pred_grid_val = 1-cph.predict_survival_function(val_results).T



In [11]:
from sklearn.metrics import mean_squared_error
df_val_pred = df_val.copy()
df_val_pred['pred'] = [np.interp(x=t, xp=pred_grid_val.columns, fp=pred_grid_val.loc[idx,:]) for idx, t in df_val_pred['duration'].items()]
mean_squared_error(y_true=df_val_pred['event_observed'], y_pred=df_val_pred['pred'])

0.1224129577933586

In [12]:
dtrain_pred = xgb.DMatrix(x_train_pred)
# dtrain = xgb.DMatrix(x_train, label=y_train)
from lifelines import CoxPHFitter

train_pred_results = pd.DataFrame({
    'xgb_risk': np.log(xgboost_model.predict(dtrain_pred)),
})
pred_grid_train = 1-cph.predict_survival_function(train_pred_results).T

In [13]:
df_train_pred_all = df_train_pred.copy()
df_train_pred_all['pred'] = [np.interp(x=t, xp=pred_grid_train.columns, fp=pred_grid_train.loc[idx,:]) for idx, t in df_train_pred_all['duration'].items()]
mean_squared_error(y_true=df_train_pred_all['event_observed'], y_pred=df_train_pred_all['pred'])

0.10437839606290383

In [14]:
import pickle
xgboost_model.save_model(f'{SAVE_PATH}_trees.ubj')


with open(f'{SAVE_PATH}_cph.pkl', 'wb') as fp:
    pickle.dump(cph, file=fp)